In [0]:
%run "../../utils/00_config"

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Silver sources
fact_orders = spark.table(
    get_table("intermediate", "fact_order_observation")
)

dim_item = spark.table(
    get_table("intermediate", "dim_item")
)

# Source validation
print(f"Fact order observations: {fact_orders.count():,}")
print(f"Item dimension: {dim_item.count():,}")

display(fact_orders.limit(5))


In [0]:

# Player status and offer visibility distribution
display(
    fact_orders
    .groupBy("player_status", "visible")
    .agg(
        F.count("*").alias("total_orders"),
        F.countDistinct("order_id").alias("unique_orders"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .orderBy("player_status", "visible")
)


In [0]:

# Filter active and visible market offers
eligible_orders = (
    fact_orders
    .filter(
        F.col("player_status").isin("online", "ingame")
        & (F.col("visible") == True)
        & F.col("platinum").isNotNull()
        & (F.col("platinum") > 0)
    )
)

print(f"Eligible orders: {eligible_orders.count():,}")

display(
    eligible_orders
    .groupBy("product_slug", "item_slug", "order_type")
    .agg(
        F.count("*").alias("total_orders"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .orderBy("product_slug", "item_slug", "order_type")
)


In [0]:
# Calculate price distribution by item and order type
price_stats = (
    eligible_orders
    .groupBy("execution_id", "item_slug", "order_type")
    .agg(
        F.count("*").alias("total_orders"),
        F.percentile_approx("platinum", 0.25, 10000).alias("q1"),
        F.percentile_approx("platinum", 0.75, 10000).alias("q3"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .withColumn("iqr", F.col("q3") - F.col("q1"))
    .withColumn("lower_bound", F.col("q1") - 1.5 * F.col("iqr"))
    .withColumn("upper_bound", F.col("q3") + 1.5 * F.col("iqr"))
    .withColumn(
        "quality_status",
        F.when(F.col("total_orders") < 8, "insufficient_sample")
        .otherwise("evaluated")
    )
)

display(
    price_stats
    .orderBy("item_slug", "order_type")
)

In [0]:
# Attach price statistics to individual orders
quality_orders = (
    eligible_orders
    .join(
        price_stats,
        on=["execution_id", "item_slug", "order_type"],
        how="left"
    )
    .withColumn(
        "is_price_outlier",
        F.when(
            F.col("quality_status") == "insufficient_sample",
            F.lit(None).cast("boolean")
        ).otherwise(
            (F.col("platinum") < F.col("lower_bound"))
            | (F.col("platinum") > F.col("upper_bound"))
        )
    )
    .withColumn(
        "price_quality_status",
        F.when(
            F.col("quality_status") == "insufficient_sample",
            "insufficient_sample"
        )
        .when(F.col("is_price_outlier"), "outlier")
        .otherwise("within_iqr")
    )
)

# Validate classification
display(
    quality_orders
    .groupBy("item_slug", "order_type", "price_quality_status")
    .agg(
        F.count("*").alias("total_orders"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .orderBy("item_slug", "order_type", "price_quality_status")
)

In [0]:
# Materialize Gold intermediate table
(
    quality_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TARGET)
)

print(f"Table materialized successfully: {GOLD_TARGET}")

In [0]:
# Validate materialized table
gold_quality = spark.table(GOLD_TARGET)

print(f"Total records: {gold_quality.count():,}")

display(
    gold_quality
    .groupBy("price_quality_status")
    .count()
    .orderBy("price_quality_status")
)

In [0]:
# Compare in-memory DataFrame with materialized table
print("In-memory DataFrame:")
display(
    quality_orders
    .groupBy("price_quality_status")
    .count()
    .orderBy("price_quality_status")
)

print("Materialized Delta table:")
display(
    gold_quality
    .groupBy("price_quality_status")
    .count()
    .orderBy("price_quality_status")
)

In [0]:
# Inspect materialized price quality by item
display(
    gold_quality
    .groupBy(
        "item_slug",
        "order_type",
        "quality_status",
        "price_quality_status"
    )
    .agg(
        F.count("*").alias("total_orders"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .orderBy("item_slug", "order_type", "price_quality_status")
)

In [0]:
# Validate logical key uniqueness
duplicate_orders = (
    gold_quality
    .groupBy("execution_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_orders.count()

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate orders."
)

print("Logical key validation passed.")